In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
print("Path to dataset files:", path)



In [2]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

class LoadData:
  def __init__(self, base_path, batch_size = 32 ):
    self.base_path = base_path
    self.batch_size = batch_size
    self.transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomRotation(10),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])
    ])
    self.train_loader = self.create_loader('train', shuffle = True)
    self.test_loader = self.create_loader('test', shuffle = False)
    self.val_loader = self.create_loader('val', shuffle = False)

  def create_loader(self,split_name, shuffle):
    folder_path = os.path.join(self.base_path, split_name)
    dataset = datasets.ImageFolder(root=folder_path, transform=self.transform)
    return DataLoader(dataset, batch_size=self.batch_size, shuffle=shuffle)

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from sklearn.metrics import classification_report, confusion_matrix

class ResNet(nn.Module):
    def __init__(self):
        super(ResNet, self).__init__()

        self.resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        num_ftrs = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(num_ftrs, 2)

    def forward(self, x):
        return self.resnet(x)

    def treinar(self, data_manager, epochs=5):
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.to(device)

        weights = torch.tensor([3.0, 1.0]).to(device)
        criterion = nn.CrossEntropyLoss(weight=weights)

        optimizer = optim.Adam(self.parameters(), lr=0.0001)

        print(f"Iniciando treinamento no dispositivo: {device}")

        for epoch in range(epochs):
            self.train()
            total_loss = 0

            for images, labels in data_manager.train_loader:
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = self(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

            print(f"Época [{epoch+1}/{epochs}] - Perda (Loss): {total_loss/len(data_manager.train_loader):.4f}")

    def avaliar(self, data_manager):
        self.eval()
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.to(device)

        todas_previsoes = []
        todos_rotulos_reais = []

        print("\nIniciando avaliação no dataset de teste...")

        with torch.no_grad():
            for images, labels in data_manager.test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = self(images)
                _, palpites = torch.max(outputs, 1)

                todas_previsoes.extend(palpites.cpu().numpy())
                todos_rotulos_reais.extend(labels.cpu().numpy())

        nomes_classes = ['NORMAL', 'PNEUMONIA']

        print("\n" + "="*40)
        print("          RELATÓRIO DE DESEMPENHO          ")
        print("="*40)
        print(classification_report(todos_rotulos_reais, todas_previsoes, target_names=nomes_classes, digits=4))

        cm = confusion_matrix(todos_rotulos_reais, todas_previsoes)
        print("Matriz de Confusão:")
        print(f" Verdadeiros Negativos (Normal correto): {cm[0][0]}")
        print(f" Falsos Positivos (Era Normal, previu Pneumonia): {cm[0][1]}")
        print(f" Falsos Negativos (Era Pneumonia, previu Normal): {cm[1][0]}")
        print(f" Verdadeiros Positivos (Pneumonia correta): {cm[1][1]}")

In [ ]:
dados = LoadData(base_path="/kaggle/input/chest-xray-pneumonia/chest_xray", batch_size=32)
modelo = ResNet()
modelo.treinar(data_manager=dados, epochs=5)
modelo.avaliar(data_manager=dados)